# Project 05 — BROKEN notebook (debugging exercise)

This notebook contains **seeded bugs** centred on link functions and prior widths. Run it, read the diagnostics, find each bug, and fix it. The clean reference is `notebook.ipynb`; the answer key is `BROKEN_BUGS.md` (don't peek first).

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
RNG = 20240601

In [ ]:
from data.generate_data import generate
data = generate()
x, y = data['x'], data['y']

### Model — two things are wrong: the prior, and the link.

In [ ]:
# BUG 1: absurdly wide coefficient priors. On the probability scale these
#        imply p is almost surely 0 or 1 before any data are seen.
# BUG 2: modeling the probability DIRECTLY (no logit link). 'p = alpha + beta*x'
#        is not a probability: it leaves (0, 1) and Bernoulli(p) will error or
#        the sampler will diverge wildly.
with pm.Model() as model:
    alpha = pm.Normal('alpha', 0.0, 10.0)
    beta = pm.Normal('beta', 0.0, 10.0)
    p = alpha + beta * x          # BUG 2: should be sigmoid(alpha + beta*x)
    pm.Bernoulli('y', p=p, observed=y)
    idata = pm.sample(draws=500, tune=500, chains=2, random_seed=RNG,
                      progressbar=False)

In [ ]:
print(az.summary(idata, var_names=['alpha', 'beta']))

### Prior predictive — BUG 3: this 'check' never looks at the probability scale, so it hides the pathology of the wide prior.

In [ ]:
a = np.random.default_rng(RNG).normal(0, 10.0, size=5000)
# BUG 3: histogramming the log-odds, not sigmoid(log-odds). Looks 'fine'
#        (a nice bell curve) and conceals that p is pinned at 0/1.
plt.hist(a, bins=30); plt.title('prior on alpha (WRONG scale)'); plt.tight_layout()